# AI-Native Quantum | MetalTech Production Scheduling
### LIVE EXECUTION DEMONSTRATION — Seven Stages, Cell by Cell

**Team:** AI-Native Quantum · **Hackathon:** QPoland Quantum Hackathon 2026

> **How to demonstrate:** Select **Runtime → Change runtime type → Python 3**, then use the **▶ button on the left of each code cell** in order. **Do not use Run all during the live talk**. Each execution cell updates its result directly below the cell. Short experiments are real, fresh computations; the historic reference figures are clearly labelled and are not presented as freshly obtained results.

**Secrets:** The optional VeloxQ stage asks privately whether to connect. If yes, you enter a key into a **masked prompt** via `getpass`; no key is embedded, logged, downloaded, or committed. Because a previous key was shared in chat, please use a newly rotated key.

**Scientific scope:** Actual QAOA is Qiskit Aer *simulation*. VeloxQ is quantum-inspired GPU optimization, not a physical QPU. Short live runs may differ from the completed research campaign; no quantum advantage claim is made.

**Seven stages:** I Problem → II QUBO → III Baseline quantum/classical → IV Industrial extension → V Robustness & VeloxQ → VI Adaptive horizons → VII Reporting.

## Setup — Install dependencies (run once)
This cell installs the required Python packages in Colab. Installation is not a solver run.

In [ ]:
%pip -q install "dimod>=0.12,<1" "dwave-neal>=0.6,<1" "qiskit>=1.1,<3" "qiskit-aer>=0.14,<1" "veloxq-sdk" "pandas>=2" "matplotlib>=3.7"

## Stage I — Original manufacturing problem
### Step 1.1 — Understand the jobs and machines
The official two-job instance uses two machines **T → F**. The official three-job instance uses three machines **T → F → S**. No machine can process two operations simultaneously and all precedence chains must be respected. The original optimum makespans are **4 hours** and **8 hours**, respectively.

In [ ]:
# Stage I / Step 1.2 — Load fixed benchmark data and helper libraries (no solvers run)
import base64, lzma, json, hashlib, os, time, math, getpass, zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
import dimod, neal

ARCHIVE_B64 = (
    '/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM50W2K1ldAD2IiCaDunFEvOttMeiZPp7BSyRpN72vxFvzhrQ/OeNjOhRvoI0RfPoafs/ztjORIN7VemJ10uiCX6ZP'
    'IINn7Vxpvz9UEVlUinQ2bmwvo3Iek+TokPLorRYFHisxPfdkJpXzyO2ACWEF+X6TswdUeAzObyOkhBz131G2mYljcg/zACFLo8+6HK2eyTttY5kvYsR4hvZ1'
    'iC+EKQSOsQzJejSbjPjpM8jnbyWNT1a1tz/qT5RX2BodsZUfmrx5GFIkPesXAFvMEigHjbdiSvLT4XK38mCtM/1yfpNa8eL4LaCLHoDHeXQTTeuHMefGfZL6'
    'qo6QBeyr87nDT2gkuzMDrJP12KIpdR0w/sMPnFt68yfVzh16aQpE2k/zm/DS65CvkfKyzH8npmnFYl3xt/bQgBgadUp3IHq21ZtKEHrPHdXieHlFP9Y/TrQM'
    '4kLeg0CEs4a67pRE3qtI989dYl+J3MmN5jlGNtoE00OHXPuSzdfYsyao37oVG4t2S7nCajNbFlPPhS8WCSNkm3OL40meoDyJ0zGZhf3eez9aYHw1FWRMK3KL'
    'wQ68u6b4IMBisc6UBvDSbEbqaf4khQh+jUfOoZ3/1MWG1Oc5+e91WDfcCa/LA8DnjOlCHbePOB4BszzI73wEkQWWL1eFyGQKROktiyX3O0TNu5AFDrns2uH6'
    'oIuPq4n6Abs9y7duKrSCm4whABcV62zTce77FjUNdxiK81aTmLS2Nde5UgbpoWqDBcLLnOlCX4N8X4wAfoWQC1ALcOeM5/vR9w14hi27XiApQmANMJXdqbdQ'
    'X5FKhKX1/nwVwC5JJw9LZFZQnc5oSEkkWZVrxmklDQf6e/XHRU/+suVHHo3P1kOLwfJvYEXw/nZdJQ+6l0fB/kNVO0i5w3r95uM18wF6VehGjdwRjNQg5a3c'
    'ED0cID0rH17YfoJT1WqSvolvYpWKPVpIdSeVhoUMqtugLxgdK+lp0wniKwHrhe4a005sFLKHzW7CxTTNjOZHVSgIZgO5pZYduR0XoEKibWeS5rjYLig/zf8O'
    'Bf15C41XpcxAvWNKUPASb5b929JTZ0T6KlxfAFaNetnEAPJLdK5xLlmv4GVFoHI7i8piRBEOv1VmUrqexqyW+wGp6Z/2+eWqaXZ+Icax43hdJqq4QFxsd+qa'
    'begR2x4e4lT3wHvvofakpLBPLqdxkeDKaQ/AslcKoJR86LFNMLq/2hE4Y0BwcXtv6an6ncKJy/Yq5LPeJL8f89THD/Tg13MHTO4enypBMC+JpfPPblWWEZpC'
    'Qo/4r8aSkUyoy9FNs6iDRXbzmZ+VtJ/8+0LJKqyomAuqga6odj0eRQc9SAJ/N8OmsN9b8Il2pXM0H/ddU9a/3HNysBBik5WzQSug9Hn6FJhZH4sLahnKzYdh'
    'mrxmLKv9s71Z81uZmcNT9GAtt9FDubj5m9BwltwraEPsRDBYH9dcLlCC0eyH7n7TpyCp3OI2QMQCvlszQvUZbstkowyOXg4xwvJFUhNz8Am3UVI5GnAZegrK'
    'iK5zPzmPPr1l/Z6bT+4XCkvbhVmQJ5r82xYHeHhRL4ZZMRXZZgpAgRaCVXwII4+IXa9KBeT5JAE/mFFV8JOvpc2sp06vEW4KlDdgJwgNBL30mBnPDoNhvMNL'
    'lLgq1+5lyDODsdFmIrDFEj+i1LpadwMRuGNXVXm3rpbb4QDCJSbbUzbWgm7FEKBhADa6GwHMjtTLkh8cpUkdWJjaZXz+p0OyAaOtB81kl6eNqWJaccDxUfh1'
    'YCly83O5lOb+l2QsLU8/WvOjfvQV81mXt/G6zR15m88YHDdRHOEVPKdrRSyCNy5STDDNSYzLadrAZpLNsoeHKAnBeJ16N5lFfIsjwXTk7QP3XQM+pwnydNPd'
    'WeEXZ9+vOxkimpVpPvrlHoIry+pDX7mP/JwDrF41RRZKJFrRNEN6bnCUtFtvs3UFAKRSSppcTjFNdIUUYbreYmY+16wNViX1Db09uTYuUkvaLWoEi5Q/tDig'
    'gli4AzQDJ6GiKxj/LRcDssDGEFtfWw3HNTq0n15rsLw12x7TV1T3p6agcTBKSReG21JCZP6tU3024SPJ9bSId/rDtOn+K45Jm/q3r7uZTFV6t2xkdC/xtYMI'
    'LPswRM/BxfhiaxYw1F/eBGED0734eIwgzGpAfPNOmxDRhPsePwBTLl+TjcKtRq+m1e5US2St9T9Lxo564JMxOcNqtUdIBYUwaWqGdp6PC5FDkr47jwDUlAmX'
    'nILPpbst3fLn1QRP4qfsiH2E+aftFrgrUZ+OJHY+po7ubZW3VbP25FTx6mLHQC+yzu2uhegCKqUn40HgFwFMCZj6i7MdrhbS+N15Utg95l6vySGRaUMBJkGl'
    'kI6V7oytvP3j+0qYLYtMwzJpjIdMTxdcRq2shP3dvn2IBobrGySCb/gwZjQE+Yy5wJGDRrlcNuq/sfGqDyzdjlvWnMPDpO9M5wnghCFxKSOnj8Pa/1a0sAkN'
    'f5YTlKPyYxr0f+uhOjil9DX/Bo4Q4SZbi/7fG9QsTJ6FqVWScY4vna6A7jK9aXrS6b46KxgcFnSTwyAIhzGB3Rvey6Y6frG/t6fRidcrWRiJ+xebiRNUi1m2'
    'OgcE/IvjHeVDWO/5IqAUx72ItVMq8b84VmYm9JbABjRJBAOutEmizAXb6lapzN8cZjwfRHSfFwaAVH0XlApwiod8Ofw4RY33gYib02zraeaM+TvIO/x4i+lx'
    'kSgH3xMXFTBm4Bg4JXuogvzDCYAPptYuQ221qdwUzZ3QxcDW5CGqI1ZWvncgCGBC8M00r96ijPF6FCzC+YyDq+gJdDrs98S4JcvLmvAzugucze1S7QjQ33Wl'
    '+Q9Udsu1tDs4poWygD/MdtI98qh8YPGXZIaozlLw8fiVvzhHD5sPps1bN8CtVjv/ePdpG63ZOOMOQ1BIsb4InOVslVEFM2mCCDExbDyX25PgJpA9JUaccxRt'
    'gMytBHOz13ihFK7DRKwTnFCiMCriqj2TS3qUOwxKxeqnUw/P0bHXmWNboeRscqLVXvyDpVqkLLoahncod3kCsqycmnRNQvm5UkQfNl0Fji3FGE1qkxgC8klR'
    'IWEsRfb05CvXK6+Xlc6vuDMoGNvcEoMpp/iX2KAgwyTru7Te+hck+FwQ23hYCgzXb93RMp53OwW4aPvSbP65v6ZNV+bsXwmmODq2geB/wO+w5uj0ze1sF6pa'
    'DbEfzyYjAk5eLuXGkmmshDthKCp4uv5zgPZYW7M1GcB1OTsLR1HWgUVdt0GoNWmqBdtDw1zT6IA2sSFFyzkzqWLcb+TcFYAv50gm/D2/gH+eDcRQCkN44hxJ'
    'HSVIb1aAazhekLHFr7j3q+sed4dNrDwWR8AEzQc6wkEVeJuNZyCKE2u9OSPg1vLsOgcLJg8cIPgZTJQk+Qpfvqlwv8sE4lBfRKXHsWibB5Dlq9UuO6KLooRJ'
    'ujsbfo5+AOAl3a3s/RuGVlL4EbLP+VEjccY1ewFxYkZYZOhOFag12BiIGryZfgOxAqfO9WuuFAnNehDru6s2MiI73eWfZ1xEQn8jGDVffWlACL7etVvhaS1i'
    'AJcRh8Q7fNOIfJ8QVlNEzXDr+/OeElyFERMbqvg+5iKzMD1Tuzd3noH8CWEQgk9BTGeWuq8j6fXwGluIdq2/z6yle9hVbcPgTMhl1WQvM4+nNrFTS1uIruYk'
    'ZnqoFZy2G6h6QT2Sp/D6/kIOFErKRihD15fPUHgtvj2XLGCHl7xxozUUY1SuyHoxMwhrxUfibekWrwx4hJH+oXQlIejmLmQrIJS2XudF9aOsUekBxl6K8A/E'
    'retU8Rj/CIjrpvOSlzrlOCJmGW3ZljReqV959NQIXBAbgfvYjcFiNVXvVOOlVg5NFMIsaoQ7x9n9izNUyd/CmcWPnYSWk4betnh+M6NyRb1QycnGED9GZkZ9'
    'WPCr7XUzU4cH5BmCiU3Cm+Cdh0BnA/ckMW4bk0aqkYZluWdfpxt3vV+AZHnG5YkL3YKuqFqwdhaWIa8dMLqR0zOlyVq0Rbb6xh14HVv2640S9t+whpHqv3Sg'
    'hrKk6Rl6IOtojmhOUnJcVPkS9lwEF+Vr9dgM2CdR2tdbCSelWLLZA2MYa1XAU/jT7x+W789WshH71+IvKmCZDCxPUthvpwEzUbOxSkWPgn92rle0dHHd2ENg'
    'vwXPxDGnTf5dcjDIQzX6f4zMcZ9Nsp+Ma/3EwcMFxxfbgDO2i9hD7xJyWujTJDK80uf6U1GKYnekp4u2dwogzAxi7RGZzJn81mjXvE+wSOczA+QmcDh3c71s'
    'vXDGBbR/rzQDi1XQh5TYrBUdhFu0CGasJrwK+BePos1/nrw38rwcHv3CHZZNbibLgnHwE45Fv828WJJaxc6b0Tg8KzjttDklTJX9Ni6pG7MLbWVkrMbinfWu'
    'k36cshhI6F0E5nI2fG2J9aqZyUvHMS9xqztKl18/tayFoOLT8zYpXcGGXQACspcKkA8xytHzM2tp5N4ZTS2fVMVlhbb+VVRlnKxH9qE09GVMjmaWetprN32+'
    'YBOrpjBnkixbe2iMrMkd024q7jFp0XgujlbrNvW9uYjgaP/f19G2S5KZVTGwOFBaFRoGKkHyuIkv/CrIA6Mf2YsifgzV1Wt/AWqzpmAUN0VyRiSq8KjIc6G5'
    'X5lyJbst2VKlucXqJdH7XLMhdwTr/BrOjgrfT7zS2Qs6BAYwY1EN9g8fVTElF/ZE4EVOAJyfs1IwGexlT0gam5UnJJAdnKWN3JoMzMDtsLEwYubrXiB5GZh4'
    'pK1+t+cZCTv5u1v5vbn1hVGU32Yxt7DQ7rdrXvVhSLb211MjTPVGta8au5uq/v7GIXoAwefzzPHqwVa00UHrBL3+fg6d5abE4b4ZNS5/6BERzoCLIiLgbhgK'
    'GVpx2ITd/woA2DS5toehy35rMkhHvRb9o7IOl0jqn+bY6udgIF/JgVJxmMMouTC3RTLr6qunQaqW7AE7IKyizpmeVzOYV3G7Y+arRwEqzUer+fiXGSlLH941'
    'LRcxWAheHObKSQUUcadVYD92mIsgoMQxjhT3S8OaTu4X45y6JqNkLzLuP9n9GtzXhaC7L2Iac+n7j+it9rJe2WqEOcX7JlfaheVTxYSdLk+vxxxP47EQbrLg'
    'cqzK9jp0YtGNoIaWQbUKCihGU7g6WarbAFnuxJ0lJvYZaP7hj5+y7PpGml5SxZ21Ksgl4KcNCGY2Qhb2vtrdBq4pbZzADWGa1BauJpRHUDnG3GNiInPLvV4S'
    'w+Ug0onPx67dvSbCKQcKpBUlsNhI+eSwW+x9Vy9YKKlcxz+BO9Cy7wn6VEa3nnrP4onlQ1/sESsHie0ZKHdK+oS9F8izpuEpNN3iicKT4p65fuKXt7RHo2G5'
    'qGma1sJCtZrCt4Pcn6GCHmNkbwAdgn6xku9WYfrlAx6/tbUsYhsmK6k1x9gw+wDlBCHMK6yl6vXV5GAjwoZADLKufuLqIZaxHQujJv/1tuETaXltBjwci9iD'
    'ooh8K3/4oF51Ci0nbw8FbGEkHj11WvQoBkkguHc776q5Jg1guTKSr1phPvaV1PL7dTqva4VuH53Y16HV0PSqUK5ESBL9qz3v3PdHF3afDmVJ1cF6wh5oJJPD'
    'cHyEWy8Ebs5LtTXz1yRaD3a7x5fkc+1jhlNlLFya5ExIEk2nj2Qq1tMwUhBHDPFyW7U4P4/6wHujaSeyWo8F1Y2lVywLvJFD37Afvc5UYg5TToyXA48uSkXt'
    'EIegQLNTmktwjfEPYvqvJLu6rorczVzUkQrCPlFIZcYBnbaDCEpKWdLo5UpAOP1JbvWrP8+hIr7+tIZPLZIzbvfK4BVxriZ74SuK4m5Wj8n8rBQMGpTDPiIZ'
    'KWJPK/7SEll32f1m/S2Xq3p5rw85Vkr8v6ZvWkSim8Jn0EoD5L+FJnqCP3RSMNtfBMKJBa7mx+W3ipwVeFY+VNRDWjMbD9PXlR8wlKhiT6snoZ2ju8gR+rb0'
    'jgPCk00IGkY7IgbjPmd5asEl4IEyaXbJ+B2gU391ppPl3ODk/3+FKQ6UKkEabTK1O467mmeM1lM4Un5vXpkxov3++FrnGPw3imK3Kl8aYV69z6hcRmH9tBMU'
    '58NmkzyZUCBPXpjzpOEo83kX8tBIfzvLQbw3yuzPGYNpZ8be1Ec70rv9yM5Plw2cJHE6JgNumFxAVlPO066QxeRzviFepddG16uGD5kS4ASVxyjvTQH6armF'
    't0SvUBARUg0E2pvaUM+KDL8KDIp6zcIuVgimZWfOVjbV+/pAwFKpy9r5V129vdZFa7/4nTVdQimiMIpM1sbFs0wn7lZeRpdjOV19RhN1OqKHjJBZd+mDfbmt'
    'LDUS0gtAl+M3IXkfk8VvcCNCx6oX5BDwN+LZZnD4BgsM2jjRhG7UYBz56ELV4KRXh7jDeJp451LFBlvdfnbXzaUd6G0yRMgefKIBzyVfGDNeHE8bbI+I2pNn'
    '8OiaET7gUyDiQ3TrPYK+fNbX+Hx9PfOZZsgG7prRabjLdJBuAq2gKX2yPikmP+QZzOVe+BFJp5hQAS8JuN8Aeo48Y57nGXwzoLLth7z+r4ty/9rw3zDmrOwa'
    'Sneewe4xAl8xgRyAfpAPuxIQxcVZJPc+3tOY6+PDPMn342cXOqdghprbJLvwJMBX4Fnlo5/HZ9Ng2TyRL8LpsH4MlL7rwR/Txum6wlpJ665c/xd3ZVAx9Bv1'
    'F6Zq1lB4ZIzkKm1Zd1v674asyf9qYiLz47FkUQY0d01Mk4bax5UPBe7TpSsbjqf0LcAaGP4QHpSns9ESEsY3k0ZJ5A6KtR5L53gjrSx5tica0TFPTClumQut'
    'NMqBFwdf4eZzouwyCUAJaWpHW/tsLqHDJgeYmnVo1kmUM5BqeNLPbOFCloybyiUIXPHyH1CdlyyXAyNKNBSrQwmqrLP11xr5XCIS7PmkmUgaEx49DWQlIZ6T'
    'l2Wl/DBodoWLMoE5GFU6269/4l1qEiYEp48kLvVA+DzvU2S+oLJiLdghhkRwiWJjMLOOU5aOqLjdZa0DYmWdrW5/39egUu+szJUcn2MCD4Ym+/nkjYPMd4K7'
    'Kq19tCop+6oCC8APtP3MbMQJDHJANPTOwZcj+PnfSJ3fsErKgL4ntk0hCV0q3n8t6zFjL+vBQ4UBVlu64XOJgExrOdvu9AGd7YDVXz9+bir7nyfYFyOZnZ4F'
    'hnjW1cvFUD+/DuvJ8f+XqBr12lt7WadcPtiSpSdvK9c6Wl81Ylnz+5uLvvQpC1DydGjixaGqGQtXUJCPXGCCDx8t9mwVVi/xax0sIauqXvjMKz9OnVlMzdCI'
    'MTPPmSq0bLO+Rp1XSR43NB31w0kaj3bYv6la1LUDMmuzZXb6/PIg7oS3h4K78mMA10+dZkRn9+M7EhjCP0AKkdM3a9MytOUv8wRWz5sNxlFM+w/9bAixJKOM'
    '0qilD144TBtkw50xDr/TecJvj7TUIwk+tFXg6jqAqptTrEu1YkorhgBLRIoFOqgelL0BYP1MBCMXZ5C1rLoqQxEWg7neImWExkMfZxVqq7OaXD1LifxRP9S/'
    'dVIKNfZEVOZihpUfGSNu59YJQT8raw432dknHQ3M+qnABTIcEAUhgRPslf83JrgccqGnn2WMISIlOwc2QUAgtV03L9R2NkFNfT4rOCivn1yx9B6B2IY0/H14'
    'TyQ04PbLEvv+gvSpaMX1jZOm9Uts8vnCvI4gSKvEN/1SHJ8vnURpbbFdLFhD4Jh87wEdwUmCMgtyXezrSCg9Cp/cVH8J3IiSGL7OO+r9Rh+wrWohVouFaA5Y'
    '6ZAtutI2+vy+HAnGE+9vziLQ6Yq9ubJIWwNC3z4Uqzm0J+PqzYHuuKaHd/v0hQanNJb3+/BNm0Tzza+BZSqExxw/t6pLIkiRLFJEr+Qmp9Ob368i7C6ShSB0'
    'NfGMjNw33Ukx2nzk6qYinx7f7duJqM6LYbPLniTxI/F+z0tAYWEmn1NU2162XEKehEUlxb4HI0yFtrvnphWFkY+wZvqjkwsWLYpUngY1e5aQa75of/8/SiNs'
    'jLAiG5p6Q8uDEVIEdcz0mYDn5pHnwfoVutnWko/CQvuoctu6TnD0j/xKZIOYj2o+EaxHdb2xYKSmEu/zgBSjUSHnHmcMBRkDyaN2FTHqbSwbiXlIvc5FU8u8'
    'ORJcZtu55A6a0dCqBW11bsFYTTcBDj6rkUYRLIy5OChyipiUXyuisENurTJZQKkMzeAfzpMDMZreKHj58WCwwENUyJEd5u6Qcrf80BFNOxJQZePgAtX50kxg'
    'Pt+VJrsTjnma+NfQxcdsYUdvrbrPhkVM8wWflSOGtPfBpLnVGK6UbwaWaDN0zUfwmbVS8UiVbXy4wgQgiBvRNyqKKc4gxbaGdXdMHKLt+81uOiUmI3SGYE76'
    'kw5JM/NGtfJkAFgP0rF0DAlNeQpWHYQcob6OY5oL1ikBfsa8MThC4T7bbBNkRkJcDLfu+KhiTOK/nOOztBbJwd05Vh20vbH7uHxUKnnlfB1I6DzCeI1cfbi0'
    '/+IdN4SeTW1q/NqxsC4zWDxbsx3mBdv75lBgKHTibDB0R+EhYD8ogMkjZZ7eLHNC8DPEdP46WjLzykdQ51Y1IMDg0NxbYKOSKEf6LYgDWB/SbjjIJZgPojUD'
    'o/ZjSI41unmVi5OnF9rAwMSkxYfgm2zCus9rD4iqOpJWvEBwqduzPz/rCmac/eralkA0UppZGqV1AicsqJFbjf5a//iC+sX5kqi4G70EXSnHcyzuUyaQWp0T'
    '5yUS9y6P9gMoKeRGbPhNI8Z83xlpc4LSeQvfT5a/yfOJnJcgsA44TcvcqMbx+eLjIv8qIwEWc3fdTA3uug+oAPpiyvCCQu/9e9m8O2QrgCGiEpUhVcuwmlqI'
    'uGOHNvt11vI5uFxy9g55/fY3bsiuED8+POw6wP1lm+XUIWvFwzj/J9rZauYyq8jjQl/QNzTe939keNtnEI1YVqm1ldWzYZR5VlGsLo26O7bihC7YftW0h2n5'
    'SqQho6aq6tXcWocDNbOYkPFIyEgL5zFRraI+ZCqh/8HTirArPncdOfHE+vwE7/PD3rBdqgmyAAFXRlvmwcBeiBnvIXHQM6U8d+8p/WXZoagLVAdNyEnzXVSZ'
    'jYYI2fr4eAy5csWpy+14NpaX9PmNF0g5uvexFrKNXZzFIVqGGQK6V2EyziWIbx2+c4D/nonJpD+uMFQ2fdYlENT/8M1rdriahaT2DRU6B1mkayIPSoDhWDYb'
    'SDTQBpBZh37glsymNF4n9vQobWVGTXlMFXxN/bEc7YEIQqYD5lpi0xXyTWOFBa7NzX8hZ8C0Ctc3pAuoR64JtSdiRZ1T+hZr99a4rSLWXbCQDOTOkODouT6b'
    'TKEFPj/B+HRZ6ebplfXjbZqTsqGcpC13snRDlggTGXsmeUEq02aw4LaNpO1aU5SsH0O/u8d2dCQ446xGpXWn4aKG9RdKsDsggek11QEg53eTJh1BfwqDIYmS'
    'jRc5XWXdtbu0RhaKtnDOOjolkSZinQC+UdUk9ctozT4IpaeQwua3Ycro3/4BX6WozjxmWf3Jmq9nuNwO/R71YJZGpjXAfn8LuSo1rioqEv+ba8Lo2rPQMqx+'
    'Yl5fPt1BCIjec4+bKfXZI5Kqen9K/7y7OcLJSQZG8rCt4GYNVvlSZfx2TemuEps0EEuTYyckEBa1+mB0pacTYkD9fw1Dxo1k0lMlhrfhlS4PadoZXlOheCb1'
    '/r2+jDM6mjs82x7bqb5/BTht526Nokbj5BjQxBkAb4+zmMPVo5REpHCDBKJ0qnpGvuksZuu5WMtDYz53CDVlmbtX96oyPrdz7OhAB3DoXZZq+ccxpP+/3u6I'
    'eMGbTua1RdbS+5w+Cn/AJBtIZHxYmJm/V3CcbU+xoEZTKaH0jHmRvrnvfz3zb0VZWh20iW7Kd1vMDS3tfvR3lzxJSYeoAHVPiKZe7gfUcGaDyTv23UO/F18U'
    'd39sWn0s6LyqZuZkz5D0urSI+e3+UY1rarSOq/4nq8Nc2ZoDnXo+afaGq/oCBTpzbICamUsLSuvs/DaOkqUviDp8YqwJyNx5UkeAE6J3aS6FplJp63DSpz/g'
    '86JXsSaa9mJ1X4STko2omM1lRKj7LiVl4/ogPjkNCCINbvYFoVGL92EowUiXTB37vPuvt+IDVBKoVmryPRKUmPasVn/IXPPO37Bt0V3mVMh3jwoOvW6+6hIh'
    '+w/b+NfVCGau24uq+izHNmKBpyZ4digjP3MQqnjyiSXKJCJR9Rh7YgWRtvUy1m9+roZCwazyVOcSWOSj7IJ/WKMUyrYotVQkW8Y9XIT+NJCjeNPxYHpFXkoj'
    'uLww09F0njUwH+2R9Vz4HfAM41kD7/3M0mQPfi6wxUVHGx0PS7+FjYDFKtMB2E1A4FPeOamom2n6s3BWLzGlYgfa0j5wKHp2zhaWJWxXOAMgOs+WIFH/4Vxx'
    'rHc8D7VMxejqhdPds0JUSH8+5/xoPP+CPvEA9UfUmUpWwKHW2rwFGRXBcAShMd7NnjOREKXuSv8CNr/+Fi+CiOGZeT62RcwUbXLD4GnSpEokE82xwhEUDxFQ'
    'r9JYHeJS3109bWimLMUFh+u/BKceYJ/0mTZGZ1p7rOvUoRX0k4AkquzNqwPHeMcyGe4ktKEE2c2ljzBTMt6A2Vj6SxsBMk60DeMiJDfxaRV0o9B+brvJIFEW'
    'caBtysARoB0Q4oJEoe6Qk7cHxVxb14dTLj5cfBLvjU1pmb73b3Uax48dDJ/DHC6XUFNaA3Eq9uIxG+0AyXk1KTXb60lohPD9oK+TybeB5bxedACYP2z5tGmF'
    '4N3N3lnF6nQfUThvA1YEoWbXB8qyEUiphHt5Qxv6h6+ayYWY/tZ0DSVuZl1tGsWp9p7U+2DdWfI7CK99Exu4iJ9iHQxLXUqIbwm7N4UthiI173Pt2E6YAl+o'
    'XgDfe8SXYB5eork3YSqcv0jw/pjLhR5pRhPdlD5z6/yNGiftopmFhUHRHGbJYG5JnkjkSfZuwIGTYpIixXXbaQnLSYFo+u/HR9tjTssMvbPh7NBxKAxQ72ho'
    'MifwjRnS1Ze1LKfI+uWoQ10FU0kpaqCTOq/PE/0imVtSe64+XlgBXWvnEbRJ+68A9gfB7Hj+0zeYM/fs9BSyRv+4QmzAmzZ2VoSsvfjE4LTPNnpSykXniwK+'
    'gHgYa9UY3Qx8I6a3ay765DicB7YEU+Ni3mJnOoficsjTl8mYGMLEuAsy4a4o53U44evjlJ336Mc7n67S+lQ6hO1Rqop4BgLM1JGe1fBztQKRzO5Qt1GbKqm6'
    'f99J7BSX+kMAv+nQH7yLqSJ5TqxyRJnHu1iA6pYWzz0gG3tUSMmbz8sB1DBtLDuPU+9lVmrHMjAKPedUyGuzM1Wh3yhd2TEVu24kBp5+bfaUrODYYcCOhdDb'
    '8jy1BI8QykE7jl8ZUeRqaMmeTj5DwRfjlK2fsvZEOiNMAzSYj7ZqP6Eq1/9jcePn6XMzVWC3Gh7NmlslAW+zeCgSULcglIw2D/+f38OE95RmYCsfble/SI9f'
    'jZcpWEQwvfhqVA8B9OaZT4CF0xMBx5UFJuxsWV+sp9IQ8RM9d2qeNacDPKYN2I11Wr4IQlUHqUTWfWpFVozpK3FLOlkAYoPqYL99go2Xp330JsV1UghS50ja'
    '/xfW2880fLIgpicTZhxf9g7ICbcFi0yVqdvkILRPwESnygRbClKStzqp+vbNRTkIKPI1gP/FPCZOBq8mW1Wo24xkE2SGOTiStEFVf8mi2DiiHUIkachivfTj'
    'TRru0RfcxFiO0qcUPjzT5hP9vpngDWTYxrARPYaTha/Mi1mfTXo6e8bYt2h/AJnplPvpJF0nUSPAqFhCpxUljLwMDxUQVz1csEfQ6UUGeyWMhGJgpbTkMhYX'
    'm+/fEeEBrK0A25JMZqKzop73ZsHL8NtSsr2/iE23Ll2T0+nE2vwUy1kpWeWvzYxZbe1CuuouIsJSHniit7HalK9wxHVdHqZrJhaoNNhZNEsAydZ7/LfVMxRR'
    'c4i4kXo55Owm3jzMI3HTadHfsgbasKbOReBU6oBSMfOJQxKiiZAzXbbiLN0HMV+gX5K1AJLVMgbbdT0iVI8DWzsJxiCtbeZKw+Qn86vv1pBD7FXKzuPbTKoJ'
    'X5jxnlEhoO5+4R+nxr653xkLO+9OqeNkJKUElqU3ozWmL8PF5/flLusf8MbRmZKiejiEDOOTFTtV7Y0ZPblGYtR7kVGXasN/FCrKvx5VXOGjV/vic8L9bMUm'
    'L3e7SPdFycl9OrMpFwmKl9HMMfJMH97jAQoj2pWOcYB1zVJNkHi2V7zpiGfLMY6Gu7KhfOEArAoVddkOsyGnKyLUpkL1r3uEiiUMAp/m6pXwQMkSBZhpBuiG'
    'PbofmgvsBhwTLux8SUdd+MZsVYkAtYAd1EblY/mMf3RHeCu4udo6DqD7/hih9NWQ85nxeCGmOi34NH8sfVcnSiYX+adpypU+tTuvveQIJmnFkhOGUvcYU0GV'
    'RERcnKI1KFEZoPb0sn+4KoXbyj+wJqtXIeJg5PRfWL+cxKpVx0UdN1PBSiKraq4vjqnvMLaownhtzbp2WzNh7A8lZPfdr1iKD+fu8K/WhJxLZhR43S0+de/3'
    'pluKfMuhBPUX/O+Ub1mTNZaPa/+uTks5gEKd0IUzQYiZg7cN12ZB62icXufYc9HBoUfV8DW+PH4APckj9rLs4SrlkRSM+DA6aM6pbB601z0LRZgzlAn8lz73'
    'tQvZu3vfTZ6VODV/1e1dtJjK0LCp6oWGcdBjpV3wR9BEK2Hb/fqYm6jjNDHiWRO+KsmhhpimCxm3ZE4OPsegtxDBxF9/h1uBV6tKY1xJJCiaXYxe8kBpTmy0'
    'Zqd+MCtkXQgA8XwyQLLHl3yq6UQ40JO2LokyICtj7Im47y3lsdO6NfB76KYGwukBkt9+Bk4w0Y9wY145RtSLd9/fKyAvJJkGPXlNbAYX93yxzESseZp1meXU'
    'lLPb4SYnYmv050cWrUIbTagTALRQi+HYJ3jehK3wR2D/BxXQUTn9yPR1kJEwhseuiiEOM2vuI6avLWA3N+PLvyoaOC753TuATBO8Q2JbcqiMrfX+0fUtZko/'
    'CcxaU3Kk2j8VgZxrx+t9OaIM+9TfFfiKQVtbTgpzSf+TOfqsEeuljx7C9FbomEBBWdXp1ploaF2EhUBSu0biXIp50XPZ0vxL4mUh/cMnAvyWa+oQA7UwSWrT'
    'XzLE2EHmqMAVxoNs6uZcMDKdyb9v7yg/0OxCCTZMnrLl/0Dkmykq/UJkTzbgmSPtttrRhLsmI2dWaaZqgae25lvrifkbqujLh0Yc2bEy8RLhmoLBZ48pNELJ'
    'GGGGWzAmECJ3jC8Kan8SL+7sK1tFCN5OOBdAObBkvsp3laIjpWB1WkHNr0UV5zAGNDtelIU68bGu/Y90IFgVpYlCJNqYOOV1mnLiUlidvqW1H3/fkp9kMWAa'
    'KIaLhb2GoqsB9pvFXK26L2dChHWNvzqZWZHWNc/nQtCv7MCHWmLbN4hdlTgxpoO9Sl8QWx2hwA4J6A1rZGB8Re8Cx9HqDu/SmKGtp0saXWxfanDpC/STySwv'
    'J6Gxb6qPkcKEZGuVsTuLW1dfZJJPxLa3LfDpHL5+4mcm0mSKRfrmAI09BchMNI3GAtglRHAGSgDISEGoqewxyCl0hX5xPTBYFrZaO8CKtCDdHZ9q6eop1tyI'
    'lBUOAN22Ijvf36B948aD9ldunqNCTdCONjN6A/u8Oj98WM+mfRYbEQLPS8FIOfjFrt8gPdBpfYgPwRNuAE0LYMNDJiNEW5amn12FUt83UDTp8J7f5otyhb8E'
    'YCyX3yPy+eASkOYssnc+xrH8L2n8fWIvTaHvhqholwSu/VLwMy2UmMVWGJnwhv0IJO5Vb/LfhsIDFBxQdP2GhtSkNc8WjXieytquXQCYchbqGjdYEfAY8p0K'
    'JlYqKH/FcUf3z4ENWlVOGUVd/7xxQP3QAaJfo61DSp0LVGlfoifjSAn2k0t42Zwg7UuV76VHcnuUr1p/TSQ8FZ6flk5iJeObrG7UnxZdAzua0tcXpMdtSmNi'
    '2m/R+oEwwgojR4oLwivmqoxO3RA1rZ9y1uYVNsQ4ddVC8BFCQfIMlJ7BXM+bzQl1gPn5W9rJ2RC/XRWqZIg/ko5DRqu69GOWBn8sOLyFbfVh/oJuur03R5EC'
    '/WrEXeIHkAdLf8y65sXzPVZe1s7MSEM2oHo/yQxaw+e25NoNG9fY2S65SOjQcwM+8E9Dvn0q/J9Y7rAQXUkIG7n86I94ZTqjy0Bu9fClK89HsnE3ThYYr1GM'
    'KVoFwvUYsR7jAvO0LTzVqyx8jZzP//iHGJH/AiwzulqwRWvqM9dEzDJ9k10P1Zp+HwNI2R6fwiI51z3PRFm9NwNHZ0Gagw9t9ShMjlL7LgOuH5bWejsavUm9'
    'aPxYcI2ZevH98d15kwQV9nqX3HPryv/Hoy9kAOtTB6tzl7XYDPiFzBajA8YqP0wHFE2RRSj8XxQQR3s5VhMBcQ4Wg2+KX0yHocZDIlUZwHY+yCBRwMJthYdA'
    'REtMxIZ7EclrGhJhdQ+d8EH9vMFEsY+RLwZTuUnQJEdctY3S+xq/QnnVD4H2lQWff7VUa5/QOuaqsr85cGANAHlM7F3ahm14RrZJUYyoB01S+W4YmTkUwlWZ'
    'wBdc/O5uomVWmN6k9IqpXyKXHg5iKAAAvCNqUGnPtLysIbMwFrP3LGc/Fz0qovMjigo5qNop5t7PKjcCiKWfqA/JK6F3JTLWm6wOyPjlfJ/X5z+YmquUA1sG'
    'z2R7KmkqIsFjVvMVa/hloMDghCSJxq0NGzqFCudKgDmXH4m6j1ujYpOiPLznHc3ZnuMtRF24NV9gQwAAAADsXLDZoGjauwAB9Va3ix0AQojwo7HEZ/sCAAAA'
    'AARZWg=='
)
EXPECTED_SHA256 = '2be1e0aad299d77128c6a65bcc25f8a3572bc267382ce58299cb8788041efa9d'
_archive_bytes=lzma.decompress(base64.b64decode(ARCHIVE_B64))
assert hashlib.sha256(_archive_bytes).hexdigest()==EXPECTED_SHA256, 'Evidence payload checksum mismatch'
ARCHIVE=json.loads(_archive_bytes)
MODELS=ARCHIVE['models']
INPUTS=ARCHIVE['inputs']
REFERENCES=ARCHIVE['references']

# New computation results are kept separately from historic reference information.
LIVE={}
RUN_STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR=Path('/content' if Path('/content').exists() else '.') / ('AI_Native_Quantum_Live_Demo_'+RUN_STAMP)
RUN_DIR.mkdir(exist_ok=True,parents=True)
print('Team: AI-Native Quantum | Mode: fresh live computation')
print('Embedded archived inputs SHA-256:',EXPECTED_SHA256)
print('Frozen core models:', ', '.join(MODELS))
print('No VeloxQ API key is stored in the notebook.')

In [ ]:
# Stage I / Step 1.3 — Display the official input durations (live data display)
for title, jobs in [
    ('Official Instance A: 2 jobs / 2 machines',{'Z1':(2,1),'Z2':(1,2)}),
    ('Official Instance B: 3 jobs / 3 machines',{'Z1':(2,1,2),'Z2':(1,2,1),'Z3':(2,2,1)})
]:
    display(Markdown('**'+title+'**'))
    machines=('T','F') if len(next(iter(jobs.values())))==2 else ('T','F','S')
    display(pd.DataFrame([{'job':job,**dict(zip(machines,ds))} for job,ds in jobs.items()]).set_index('job'))
print('Historic certified optima: A = 4 hours; B = 8 hours.')

## Stage II — Engineering constraints → QUBO formulation
### Step 2.1 — Binary start variables; Step 2.2 — QUBO penalties; Step 2.3 — Reduced search domain
Each binary decision **x[j,m,t]** represents a job-operation starting on a machine at time *t*. Violations of one-start-per-operation, machine exclusivity, and processing order incur positive quadratic penalties. The archived 3×3 baseline prunes a 72-variable initial time grid to **39 variables / 181 interactions**.

In [ ]:
# Stage II — Load unmodified frozen model coefficients and inspect them

def bqm_from_json(spec):
    return dimod.BinaryQuadraticModel(
        {k:float(v) for k,v in spec['linear'].items()},
        {(e['u'],e['v']):float(e['bias']) for e in spec['quadratic']},
        float(spec['offset']),dimod.BINARY)

BQM={key:bqm_from_json(spec) for key,spec in MODELS.items()}
for key,bqm in BQM.items():
    print(f'{key:28s}: {len(bqm.variables):3d} binary variables, {len(bqm.quadratic):4d} couplers, offset={bqm.offset:.4f}')
assert (len(BQM['official_2jobs_H4'].variables),len(BQM['official_3jobs_H8'].variables))==(8,39)
assert len(BQM['official_3jobs_H8'].quadratic)==181
print('PASS: Frozen scheduling QUBOs loaded; no solver executed.')

In [ ]:
# Stage II / Step 2.4 — Application decoder, schedule QA, and Gantt visualization
MACHINES=('T','F','S')

def decode_sample(model_id,sample):
    if model_id=='official_2jobs_H4':
        processing={'Z1':(2,1),'Z2':(1,2)};machine_list=('T','F');H=4; official=True
        cfg={}
    else:
        cfg=INPUTS[model_id];processing=cfg['processing_hours'];machine_list=MACHINES
        H=int(REFERENCES[model_id]['target_makespan']);official=bool(cfg.get('official'))
    issues=[];selected={}
    for job,durs in processing.items():
        for m in machine_list:
            arr=[str(key) for key,value in sample.items() if str(key).startswith(f'{job}_{m}_') and int(value)==1]
            if len(arr)!=1:
                issues.append(f'{job}/{m} has {len(arr)} starts');continue
            t=int(arr[0].rsplit('_',1)[1]);d=int(durs[machine_list.index(m)])
            selected[(job,m)]=(t,t+d)
            if t<0 or t+d>H:issues.append(f'{job}/{m} horizon violation')
            if not official and m=='F':
                _,lo,hi=cfg['outage']
                if t<hi and t+d>lo:issues.append(f'{job}/{m} maintenance violation')
    for job in processing:
        for m1,m2 in zip(machine_list,machine_list[1:]):
            if (job,m1) in selected and (job,m2) in selected and selected[job,m1][1]>selected[job,m2][0]:
                issues.append(f'{job} precedence violation')
    for m in machine_list:
        intervals=sorted((s,e,j) for (j,mm),(s,e) in selected.items() if mm==m)
        if any(left[1]>right[0] for left,right in zip(intervals,intervals[1:])):
            issues.append(f'{m} capacity violation')
    schedule=sorted([dict(job=j,machine=m,start=s,end=e) for (j,m),(s,e) in selected.items()],key=lambda d:(d['start'],d['machine'],d['job']))
    makespan=max([o['end'] for o in schedule],default=0)
    if makespan!=H:issues.append(f'exact horizon mismatch: {makespan} != {H}')
    if issues:return dict(feasible=False,issues=issues,schedule=schedule,makespan=makespan)
    if official:return dict(feasible=True,issues=[],schedule=schedule,makespan=makespan,objective=float(makespan),electricity_usd=None)
    cents=0;late=0
    for job,durs in processing.items():
        finish=selected[(job,'S')][1]
        late+=int(cfg['priority_weights'][job])*max(0,finish-int(cfg['due_hours'][job]))
        for m in machine_list:
            s,e=selected[(job,m)];power=cfg['power_kw'][m]
            for hr in range(s,e): cents+=power*(27 if hr%8 in (2,3,4,5) else 11)
    objective=makespan+2*late+cents/5000
    return dict(feasible=True,issues=[],schedule=schedule,makespan=makespan,
                weighted_tardiness=late,electricity_usd=cents/100,objective=round(objective,8))

def plot_gantt(schedule,title):
    if not schedule:
        print('No schedule to plot.');return
    fig,ax=plt.subplots(figsize=(9.5,3.1));machines=list(dict.fromkeys(o['machine'] for o in schedule))
    colors={'Z1':'#1967d2','Z2':'#00a9a1','Z3':'#ff944d','Z4':'#8e63d7','Z5':'#d65881'}
    for o in schedule:
        y=machines.index(o['machine']);ax.barh(y,o['end']-o['start'],left=o['start'],height=.65,color=colors.get(o['job'],'gray'),edgecolor='white')
        ax.text((o['start']+o['end'])/2,y,o['job'],va='center',ha='center',color='white',weight='bold',fontsize=10)
    ax.set_yticks(range(len(machines)),machines);ax.invert_yaxis();ax.set_xlabel('Hour');ax.set_title(title)
    ax.set_xlim(0,max(o['end'] for o in schedule)+0.3);ax.set_xticks(range(max(o['end'] for o in schedule)+1));ax.grid(axis='x',alpha=.15)
    plt.tight_layout()
    fig.savefig(RUN_DIR / f'gantt_{len(list(RUN_DIR.glob("gantt_*.png")))+1:02d}.png',dpi=145,bbox_inches='tight')
    display(fig);plt.close(fig)

# Reusable live neal run: best *feasible* sample, not simply lowest raw QUBO energy.
def live_neal(model_id,reads=800,sweeps=1300,seed=20261017):
    bqm=BQM[model_id];start=time.perf_counter()
    ss=neal.SimulatedAnnealingSampler().sample(bqm,num_reads=reads,num_sweeps=sweeps,seed=seed)
    best=None;valid=0;total=0;lowest_energy=float('inf')
    for row in ss.data(['sample','energy','num_occurrences']):
        occ=int(row.num_occurrences);total+=occ
        score=decode_sample(model_id,row.sample)
        lowest_energy=min(lowest_energy,float(row.energy))
        if score['feasible']:
            valid+=occ
            if best is None or score['objective']<best['objective']-1e-10:
                best={**score,'qubo_energy':float(row.energy)}
    ref=4 if model_id=='official_2jobs_H4' else float(REFERENCES[model_id].get('target_objective',REFERENCES[model_id]['target_makespan']))
    result={'method':'fresh neal simulated annealing','model':model_id,'reads':reads,'sweeps':sweeps,'seed':seed,
      'runtime_seconds':round(time.perf_counter()-start,3),'valid_samples':valid,'total_samples':total,
      'valid_fraction':valid/total if total else 0,'lowest_raw_qubo_energy':lowest_energy,'best':best,
      'historic_certified_reference':ref,'matched_reference':bool(best and abs(best['objective']-ref)<1e-5)}
    LIVE['neal_'+model_id]=result
    print('LIVE neal:',model_id,'| samples',total,'| feasible',valid,'| fraction',f'{result["valid_fraction"]:.1%}')
    if best:
        print('Best feasible objective:',best['objective'],'| historic CP-SAT reference:',ref,
              '| matched:',result['matched_reference'])
        plot_gantt(best['schedule'],f'LIVE neal schedule: {model_id}')
    else:print('No feasible sample in this live run; historical reference still available.')
    return result
print('Decoder, independent feasibility checks, Gantt plotting and live classical sampler ready.')

## Stage III — Official baseline: quantum and classical execution
### Step 3.1 — Fresh 8-qubit QAOA measurement run (Qiskit Aer)
This cell **really runs a new Qiskit Aer circuit**. It reuses the previously fitted p=3 angles to keep the demonstration fast; it **does not rerun COBYLA fitting**. The original historical run observed **142/4096 optimal shots**, but fresh measurements may differ.

In [ ]:
# Stage III / Step 3.1 — ACTUAL QAOA Aer simulator circuit execution (no hardware)
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
bqm=BQM['official_2jobs_H4']; labels=['Z1_T_0','Z1_T_1','Z1_F_2','Z1_F_3','Z2_T_0','Z2_T_1','Z2_F_1','Z2_F_2']
assert set(labels)==set(bqm.variables)
idx={name:i for i,name in enumerate(labels)}
z={i:-float(bqm.linear[name])/2 for i,name in enumerate(labels)}; zz={}
for (name_i,name_j),v in bqm.quadratic.items():
    i,j=idx[name_i],idx[name_j];coef=float(v)/4
    z[i]-=coef;z[j]-=coef;zz[i,j]=coef
angles=ARCHIVE['qaoa']['best']['angles']; depth=ARCHIVE['qaoa']['best']['depth']
qc=QuantumCircuit(8);qc.h(range(8))
for layer in range(depth):
    gamma,beta=angles[layer],angles[depth+layer]
    for i,h in z.items():qc.rz(2*gamma*h,i)
    for (i,j),h in zz.items():qc.rzz(2*gamma*h,i,j)
    for i in range(8):qc.rx(2*beta,i)
qc.measure_all()
sim=AerSimulator(method='statevector')
t0=time.perf_counter()
counts=sim.run(transpile(qc,sim,optimization_level=1),shots=4096).result().get_counts()
valid_samples=[];optimal_shots=0
for bits,count in counts.items():
    bitindex=int(bits.replace(' ',''),2)
    sample={k:(bitindex>>i)&1 for i,k in enumerate(labels)}
    outcome=decode_sample('official_2jobs_H4',sample)
    if outcome['feasible']:
        optimal_shots+=count;valid_samples.append((count,outcome))
val=max(valid_samples,key=lambda a:a[0])[1] if valid_samples else None
LIVE['qaoa_A']={'method':'fresh Qiskit Aer QAOA sampling; archived p=3 angles','shots':4096,'optimal_shots':int(optimal_shots),
                'optimal_fraction':optimal_shots/4096,'elapsed_seconds':round(time.perf_counter()-t0,3),
                'historic_optimal_shots':142,'best_feasible':val}
print('Qiskit Aer QAOA freshly executed. Optimal shots:',optimal_shots,'/ 4096; historic experiment: 142 / 4096')
print('Archived COBYLA optimizer reported success:',ARCHIVE['qaoa']['best']['optimizer_success'])
if val:plot_gantt(val['schedule'],'LIVE Qiskit Aer QAOA — official 2×2')
else:print('No feasible 4-hour schedule sampled this run; do not claim a fresh optimal quantum sample.')
fig,ax=plt.subplots(figsize=(8,2.7));top=sorted(counts.items(),key=lambda kv:kv[1],reverse=True)[:12]
ax.bar([x[0] for x in top],[x[1] for x in top]);ax.set_title('Fresh QAOA measurement counts (top 12 states)');ax.tick_params(axis='x',rotation=60)
plt.tight_layout();fig.savefig(RUN_DIR/'live_qaoa_counts.png',dpi=145,bbox_inches='tight');display(fig);plt.close(fig)

### Step 3.2 — Fresh classical QUBO execution for official 3×3 instance
**Actual calculation:** `neal.SimulatedAnnealingSampler` searches 39 binary variables and 181 couplings. The independent decoder shows whether the **fresh run** reaches the archived 8-hour CP-SAT optimum.

In [ ]:
# Stage III / Step 3.2 — ACTUAL 39-variable classical QUBO execution
baseline_run=live_neal('official_3jobs_H8',reads=1200,sweeps=1600,seed=20261017)

### Step 3.3 — Baseline scorecard
This prints a side-by-side comparison. CP-SAT figures are **previously certified historical references**, not newly recomputed results.

In [ ]:
# Stage III / Step 3.3 — Live results beside frozen certified references
records=[{'method':'Archived CP-SAT reference','instance':'2 × 2','makespan_h':4,'fresh_run':False},
         {'method':'Archived CP-SAT reference','instance':'3 × 3','makespan_h':8,'fresh_run':False}]
if LIVE.get('qaoa_A',{}).get('best_feasible'):
    records.append({'method':'Fresh QAOA Aer','instance':'2 × 2','makespan_h':4,'fresh_run':True})
if baseline_run.get('best'):
    records.append({'method':'Fresh neal','instance':'3 × 3','makespan_h':baseline_run['best']['makespan'],'fresh_run':True})
display(pd.DataFrame(records))

## Stage IV — Synthetic industrial requirements
### Step 4.1 — Milling outage; Step 4.2 — Weighted tardiness and time-of-use electricity
A milling outage **[3,5)**, job deadlines, priority weights, synthetic machine power, and peak/off-peak tariffs enter a 40-variable QUBO. This test shows a *fresh classical QUBO run* against the **archived CP-SAT optimum F=11.4456**. The research archive showed synthetic electricity cost **$23.56 → $22.28**, but that historical comparison is not guaranteed to reproduce from one short new run.

In [ ]:
# Stage IV / Step 4.3 — ACTUAL energy-aware QUBO execution
energy_run=live_neal('energy_3jobs_H9_P25',reads=1200,sweeps=1600,seed=20261018)
if energy_run['best']:
    print('Fresh schedule electricity cost: $',round(energy_run['best']['electricity_usd'],2))
    print('Fresh weighted tardiness:',energy_run['best']['weighted_tardiness'])
print('Historic energy-aware experiment: $23.56 vs $22.28 on comparable synthetic schedules.')

## Stage V — Robustness, scaling, QAOA depth and VeloxQ
### Step 5.1 — Actual penalty-sensitivity sampling, with archived coefficients
These nine models (3, 4, and 5 jobs × penalties 5/25/100) are the **same frozen models used in the completed robustness campaign**. We run one quick new batch per model. One live seed is not equivalent to the previous five-seed research study.

In [ ]:
# Stage V / Steps 5.1–5.3 — Nine FRESH neal penalty tests (3/4/5 jobs)
penalty_rows=[]
for n in (3,4,5):
    for P in (5,25,100):
        model_id=f'{n}jobs_P{P}'
        bb=bqm_from_json(ARCHIVE['penalty_models'][model_id]);t0=time.perf_counter()
        samples=neal.SimulatedAnnealingSampler().sample(bb,num_reads=90,num_sweeps=650,seed=20261017+P)
        # Full feasibility here requires each job-operation exactly once, machine precedence and outage.
        # Reuse the four reference-domain helper inputs with the same 3/4/5-job synthetic parameters.
        ref_key={3:'energy_3jobs_H9_P25',4:'energy_4jobs_H12_P25',5:'energy_5jobs_H13_P25'}[n]
        ref_cfg=INPUTS[ref_key]
        H={3:9,4:11,5:12}[n]  # Stage V minimum-makespan horizon
        # The Stage V models have fixed H and do NOT contain the exact-H anchor.
        valid_count=0;best_obj=None
        for datum in samples.data(['sample','num_occurrences']):
            by={};issues=[]
            for job,ds in list(ref_cfg['processing_hours'].items())[:n]:
                for m in ('T','F','S'):
                    choices=[k for k,v in datum.sample.items() if k.startswith(job+'_'+m+'_') and int(v)]
                    if len(choices)!=1:issues.append('onehot');continue
                    s=int(choices[0].rsplit('_',1)[1]);d=ds[('T','F','S').index(m)];by[job,m]=(s,s+d)
                    if s+d>H:issues.append('horizon')
                    if m=='F' and s<5 and s+d>3:issues.append('outage')
            for job in list(ref_cfg['processing_hours'])[:n]:
                if (job,'T') in by and (job,'F') in by and by[job,'T'][1]>by[job,'F'][0]:issues.append('precedence')
                if (job,'F') in by and (job,'S') in by and by[job,'F'][1]>by[job,'S'][0]:issues.append('precedence')
            for m in ('T','F','S'):
                spans=sorted((s,e) for (j,mm),(s,e) in by.items() if mm==m)
                if any(x[1]>y[0] for x,y in zip(spans,spans[1:])):issues.append('collision')
            if not issues and len(by)==3*n:
                valid_count+=int(datum.num_occurrences)
                makespan=max(e for s,e in by.values())
                late=sum(ref_cfg['priority_weights'][j]*max(0,by[j,'S'][1]-ref_cfg['due_hours'][j]) for j in list(ref_cfg['processing_hours'])[:n])
                cents=sum(ref_cfg['power_kw'][m]*(27 if hr%8 in (2,3,4,5) else 11)
                          for (j,m),(s,e) in by.items() for hr in range(s,e))
                score=makespan+2*late+cents/5000
                if best_obj is None or score<best_obj:best_obj=score
        penalty_rows.append({'jobs':n,'penalty':P,'QUBO_variables':len(bb.variables),'couplings':len(bb.quadratic),
                             'feasible_pct':round(100*valid_count/90,2),'best_live_objective':round(best_obj,5) if best_obj is not None else None,
                             'elapsed_s':round(time.perf_counter()-t0,3)})
penalty_df=pd.DataFrame(penalty_rows)
LIVE['penalty_sweep']=penalty_rows
display(penalty_df)
fig,ax=plt.subplots(figsize=(8,3.3))
for P in (5,25,100):
    d=penalty_df[penalty_df.penalty==P]
    ax.plot(d.jobs,d.feasible_pct,marker='o',label=f'P={P}')
ax.set_xlabel('Jobs');ax.set_ylabel('Fresh feasible-sample %');ax.set_title('Live penalty-sensitivity test — single seed / 90 reads');ax.legend();ax.grid(alpha=.2)
plt.tight_layout();fig.savefig(RUN_DIR/'live_penalty_sensitivity.png',dpi=145,bbox_inches='tight');display(fig);plt.close(fig)

### Step 5.4 — Fresh QAOA depth demonstrations
Run a **new Qiskit Aer circuit** at p=1,2,3, using saved angles from the completed depth study. This is a sampling demonstration, **not fresh variational optimization**.

In [ ]:
# Stage V / Step 5.4 — Three actual Aer measurements at depths 1, 2, 3
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
qdepth_rows=[]
for p in (1,2,3):
    matches=[r for r in ARCHIVE['qaoa']['depth_trials'] if r['depth']==p]
    historical=min(matches,key=lambda r:r['fitted_mean_energy'])
    params=historical['optimized_parameters_radians']
    circuit=QuantumCircuit(8);circuit.h(range(8))
    for l in range(p):
        gamma,beta=params[2*l],params[2*l+1]
        for i,h in z.items():circuit.rz(2*gamma*h,i)
        for (i,j),h in zz.items():circuit.rzz(2*gamma*h,i,j)
        for i in range(8):circuit.rx(2*beta,i)
    circuit.measure_all()
    t0=time.perf_counter();counts2=AerSimulator().run(transpile(circuit,AerSimulator()),shots=1024).result().get_counts()
    opt=0
    for bitstring,occ in counts2.items():
        number=int(bitstring.replace(' ',''),2);sample={k:(number>>i)&1 for i,k in enumerate(labels)}
        if decode_sample('official_2jobs_H4',sample)['feasible']:opt+=occ
    qdepth_rows.append({'QAOA depth':p,'fresh_shots':1024,'fresh_optimal_shots':opt,'fresh_optimal_pct':round(100*opt/1024,2),
                        'historic_best_fit_reported_success':historical['fit_success'],
                        'runtime_seconds':round(time.perf_counter()-t0,3)})
LIVE['qaoa_depth']=qdepth_rows
display(pd.DataFrame(qdepth_rows))

### Step 5.5 — OPTIONAL real VeloxQ GPU remote jobs (API-key gate)
This section is **optional**, separate from Qiskit Aer and classical `neal`. To demonstrate VeloxQ, use a newly rotated team key. The notebook **prompts privately at runtime**; it never contains the token. You can press Enter to skip without blocking the remaining research stages. Remote submissions may consume quota or take longer than expected.

In [ ]:
# Stage V / Step 5.5.1 — User decision + PRIVATE masked VeloxQ token input
RUN_VELOXQ=input('Run FOUR live remote VeloxQ jobs now? Type YES to approve, or press Enter to skip: ').strip().upper()=='YES'
AUTH_READY=False
if RUN_VELOXQ:
    token=getpass.getpass('VeloxQ team API key (hidden; use a rotated key): ').strip()
    if token:
        os.environ['VELOX_TOKEN']=token
        AUTH_READY=True
        print('Authentication input accepted privately. Token will never be printed or saved.')
    else:print('No token entered. Remote jobs skipped.')
    del token
else:print('VeloxQ remote execution skipped. Historical four-run results remain available in the presentation notebook.')
LIVE['veloxq_authorized']=AUTH_READY

In [ ]:
# Stage V / Step 5.5.2 — VeloxQ helper (no job submitted until each numbered run cell)
VELOX_RESULT={}

def remote_veloxq(model_id):
    if not AUTH_READY:
        print(model_id,': SKIPPED — no API token supplied');VELOX_RESULT[model_id]={'status':'SKIPPED'};return None
    from veloxq_sdk import VeloxQSolver
    bqm=BQM[model_id];solver=VeloxQSolver()
    solver.parameters.num_rep=256;solver.parameters.num_steps=1500
    t0=time.perf_counter()
    try:
        samples=solver.sample(bqm)  # Real remote execution, not a saved result
        if samples.vartype!=dimod.BINARY:raise ValueError('Returned non-binary samples')
        best=None;valid=0;total=0
        for r in samples.data(['sample','energy','num_occurrences']):
            raw={str(k):int(v) for k,v in dict(r.sample).items()}
            # SDK may return numeric/string positional variable IDs; remap by input BQM order.
            target=[str(v) for v in bqm.variables];received=[str(v) for v in samples.variables]
            if set(raw)==set(target): bit=raw
            elif set(raw)==set(received) and len(received)==len(target):
                bit={target[i]:raw[received[i]] for i in range(len(target))}
            else:raise ValueError('Cannot safely decode VeloxQ variable labels')
            if abs(bqm.energy(bit)-float(r.energy))>5e-4:raise ValueError('Returned QUBO energy differs from input model')
            score=decode_sample(model_id,bit);occ=int(r.num_occurrences);total+=occ
            if score['feasible']:
                valid+=occ
                if best is None or score['objective']<best['objective']:
                    best={**score,'qubo_energy':float(r.energy)}
        ref=float(REFERENCES[model_id].get('target_objective',REFERENCES[model_id]['target_makespan']))
        result={'status':'SUCCESS','model':model_id,'backend':'VeloxQ GPU quantum-inspired','reads':total,
                'feasible':valid,'feasible_fraction':valid/total if total else 0,
                'wall_seconds':round(time.perf_counter()-t0,3),'best':best,
                'reference_objective':ref,'match_reference':bool(best and abs(best['objective']-ref)<1e-5)}
        print('REMOTE VELOXQ SUCCESS:',model_id,'| feasible:',valid,'/',total)
        print('Best objective:',best['objective'] if best else 'No feasible sample','| historic reference:',ref)
        if best:plot_gantt(best['schedule'],'LIVE VeloxQ result — '+model_id)
    except Exception as exc:
        # Never print network errors: response strings may contain token fragments.
        result={'status':'ERROR','model':model_id,'error_type':type(exc).__name__,'seconds':round(time.perf_counter()-t0,3)}
        print('VeloxQ job error:',type(exc).__name__,'— sensitive details suppressed.')
    VELOX_RESULT[model_id]=result;LIVE['veloxq_'+model_id]=result
    return result
print('VeloxQ initialized for user-controlled execution; no job submitted yet.')

### Step 5.5.3 — VeloxQ remote job 1/4: official 39-variable baseline
**Click ▶ to submit just this job.** Skip safely if you did not enter a key. The new results will appear immediately below the cell.

In [ ]:
# ACTUAL VeloxQ job 1/4 — official 39-variable baseline
remote_veloxq('official_3jobs_H8')

### Step 5.5.4 — VeloxQ remote job 2/4: 40-variable energy case
**Click ▶ to submit just this job.** Skip safely if you did not enter a key. The new results will appear immediately below the cell.

In [ ]:
# ACTUAL VeloxQ job 2/4 — 40-variable energy case
remote_veloxq('energy_3jobs_H9_P25')

### Step 5.5.5 — VeloxQ remote job 3/4: 88-variable adaptive 4-job case
**Click ▶ to submit just this job.** Skip safely if you did not enter a key. The new results will appear immediately below the cell.

In [ ]:
# ACTUAL VeloxQ job 3/4 — 88-variable adaptive 4-job case
remote_veloxq('energy_4jobs_H12_P25')

### Step 5.5.6 — VeloxQ remote job 4/4: 128-variable adaptive 5-job case
**Click ▶ to submit just this job.** Skip safely if you did not enter a key. The new results will appear immediately below the cell.

In [ ]:
# ACTUAL VeloxQ job 4/4 — 128-variable adaptive 5-job case
remote_veloxq('energy_5jobs_H13_P25')

In [ ]:
# Stage V — Remote session clean-up; no keys are exported
os.environ.pop('VELOX_TOKEN',None)
print('VeloxQ token removed from the process environment. No secret exported.')

## Stage VI — Adaptive-horizon optimization
### Step 6.1 — Research question: why shortest production time may not minimize business cost
Instead of fixing every job set to its shortest possible makespan, compare 4 jobs at H=11 vs H=12 and 5 jobs at H=12 vs H=13. The QUBOs include the explicit exact-horizon anchor. A *fresh* short `neal` run can reproduce or approximate the archived result; the CP-SAT values below remain historical certified references.

In [ ]:
# Stage VI / Steps 6.2–6.4 — Four ACTUAL adaptive-horizon QUBO simulations
adaptive_rows=[]
for n,H in ((4,11),(4,12),(5,12),(5,13)):
    spec=ARCHIVE['adaptive_models'][f'{n}jobs_H{H}'];bb=bqm_from_json(spec)
    t0=time.perf_counter();batch=neal.SimulatedAnnealingSampler().sample(bb,num_reads=300,num_sweeps=1200,seed=20261017+H)
    valid=0;best=None
    source='energy_4jobs_H12_P25' if n==4 else 'energy_5jobs_H13_P25'
    cfg=INPUTS[source];proc=cfg['processing_hours']
    for row in batch.data(['sample','num_occurrences']):
        sched=[];issues=[];spans={}
        for job in list(proc)[:n]:
            for m,d in zip(('T','F','S'),proc[job]):
                candidates=[k for k,v in row.sample.items() if k.startswith(job+'_'+m+'_') and int(v)]
                if len(candidates)!=1:issues.append('onehot');continue
                s=int(candidates[0].rsplit('_',1)[1]);e=s+d;spans[job,m]=(s,e)
                sched.append({'job':job,'machine':m,'start':s,'end':e})
                if e>H:issues.append('horizon')
                if m=='F' and s<5 and e>3:issues.append('outage')
        for job in list(proc)[:n]:
            if all((job,m) in spans for m in ('T','F','S')):
                if spans[job,'T'][1]>spans[job,'F'][0] or spans[job,'F'][1]>spans[job,'S'][0]:issues.append('precedence')
        for m in ('T','F','S'):
            interval=sorted((s,e) for (j,mm),(s,e) in spans.items() if mm==m)
            if any(x[1]>y[0] for x,y in zip(interval,interval[1:])):issues.append('collision')
        if len(spans)!=3*n or max((e for s,e in spans.values()),default=0)!=H:issues.append('exact-H')
        if not issues:
            valid+=int(row.num_occurrences)
            tard=sum(cfg['priority_weights'][j]*max(0,spans[j,'S'][1]-cfg['due_hours'][j]) for j in list(proc)[:n])
            cents=sum(cfg['power_kw'][m]*(27 if hr%8 in (2,3,4,5) else 11)
                      for (j,m),(s,e) in spans.items() for hr in range(s,e))
            objective=H+2*tard+cents/5000
            if best is None or objective<best['objective']:
                best={'objective':round(objective,7),'schedule':sched,'energy_usd':cents/100}
    historic={(4,11):15.5360,(4,12):14.5808,(5,12):16.6684,(5,13):15.7132}[(n,H)]
    adaptive_rows.append({'jobs':n,'exact_H':H,'variables':len(bb.variables),'valid_samples':valid,
                          'fresh_best_objective':best['objective'] if best else None,
                          'archived_CP_SAT_optimum':historic,
                          'matched':bool(best and abs(best['objective']-historic)<1e-4),
                          'seconds':round(time.perf_counter()-t0,3)})
    print(f'{n} jobs / H={H}: fresh feasible={valid}/300; fresh best={best["objective"] if best else "none"}; certified archived={historic}')
    if best and H in (12,13):plot_gantt(best['schedule'],f'LIVE adaptive {n} jobs — H={H}')
LIVE['adaptive_horizon']=adaptive_rows
display(pd.DataFrame(adaptive_rows))

In [ ]:
# Stage VI / Step 6.5 — Archived scientific trade-off (not a newly certified claim)
old4,new4=15.5360,14.5808;old5,new5=16.6684,15.7132
print(f'Archived 4-job improvement (H11 → H12): {(old4-new4)/old4:.2%}')
print(f'Archived 5-job improvement (H12 → H13): {(old5-new5)/old5:.2%}')
fig,ax=plt.subplots(figsize=(7,3.3))
x=np.arange(2);ax.bar(x-0.18,[old4,old5],.36,label='Certified fixed shortest horizon')
ax.bar(x+0.18,[new4,new5],.36,label='Certified adaptive-horizon best')
ax.set_xticks(x,['4 jobs','5 jobs']);ax.set_ylabel('Composite objective (lower is better)');ax.legend();ax.set_title('Historic exact CP-SAT optimum: horizon selection matters')
plt.tight_layout();fig.savefig(RUN_DIR/'historical_horizon_tradeoff.png',dpi=145,bbox_inches='tight');display(fig);plt.close(fig)

## Stage VII — Integrated scientific results, provenance and final export
### Step 7.1 — Distinguish live results from previously verified results
### Step 7.2 — State qualified findings

**Core conclusion:** Minimum manufacturing makespan does not necessarily minimize the multi-objective industrial score. Carefully choosing the feasible horizon can yield a better certified objective. Qiskit Aer is **a simulation**, `neal` is **classical annealing**, VeloxQ is **GPU-based quantum-inspired optimization**. This experiment does **not** establish physical-QPU quantum advantage.

**Original archived VeloxQ campaign:** 4 successful remote jobs; 3 exact objective matches and a near-optimal 5-job solution with 0.163% gap. This display-only historical result is distinct from any newly executed remote jobs above.

In [ ]:
# Stage VII / Step 7.3 — Live/historic scorecard and automatic evidence ZIP download
historic_velox=pd.DataFrame(ARCHIVE['archived_velox'])
cols=['benchmark','binary_variables','reference_business_objective','veloxq_best_business_objective',
      'veloxq_feasible_fraction','veloxq_optimum_match']
display(Markdown('**Previously completed VeloxQ study (historical archive, not a newly executed run)**'))
display(historic_velox[cols])
historic_velox[cols].to_csv(RUN_DIR/'historical_velox_comparison.csv',index=False)
if LIVE.get('penalty_sweep'):pd.DataFrame(LIVE['penalty_sweep']).to_csv(RUN_DIR/'fresh_penalty_sweep.csv',index=False)
if LIVE.get('adaptive_horizon'):pd.DataFrame(LIVE['adaptive_horizon']).to_csv(RUN_DIR/'fresh_adaptive_horizons.csv',index=False)
print('New live computation blocks recorded:',len(LIVE))

# JSON export refuses to include any key/token/config/environment dictionary.
clean={k:v for k,v in LIVE.items() if 'token' not in k.lower() and 'key' not in k.lower()}
archive={'team':'AI-Native Quantum','mode':'live cell-by-cell demonstration',
         'archive_sha256':EXPECTED_SHA256,'created_utc':RUN_STAMP,'new_results':clean,
         'historical_velox_reference':ARCHIVE['archived_velox'],
         'scientific_caveat':'Qiskit Aer=simulation; VeloxQ=GPU quantum-inspired; no physical QPU claim.'}
export_json=RUN_DIR/'LIVE_DEMO_RESULTS.json';export_json.write_text(json.dumps(archive,indent=2,default=str))
summary=RUN_DIR/'READ_ME_FIRST.txt';summary.write_text('AI-Native Quantum LIVE demo evidence. New runs are separate from historical benchmarks.\nNo API token is included.\n')
output_zip=RUN_DIR.with_suffix('.zip')
with zipfile.ZipFile(output_zip,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.rglob('*'):
        if p.is_file():zf.write(p,arcname=p.relative_to(RUN_DIR.parent))
print('Results ZIP created:',output_zip.name,'| bytes:',output_zip.stat().st_size)
try:
    from google.colab import files
    files.download(str(output_zip))
    print('Browser download initiated. If blocked, download the ZIP from the Colab Files sidebar.')
except ImportError:
    print('Not running in Colab. Evidence is saved at:',output_zip)